## Write to and read from a Delta Lake table

### Write a Spark DataFrame to a Delta Lake table

In [1]:
from pyspark.sql import SparkSession
import subprocess

spark = SparkSession.builder \
    .appName("DeltaLakeDemo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

delta_path = "/opt/spark/delta/"

In [2]:
import os,shutil

if os.path.exists(delta_path+"table1"):
    shutil.rmtree(delta_path+"table1")

In [3]:
data = spark.range(0, 5)

(data
  .write
  .format("delta")
  .save(delta_path + "table1")
)

### Read the above Delta Lake table to a Spark DataFrame and display the DataFrame

In [4]:
df = (spark
        .read
        .format("delta")
        .load(delta_path + "table1")
        .orderBy("id")
      )

df.show()

### Overwrite a Delta Lake table

In [5]:
data = spark.range(5, 10)

# Create a temporary view from the DataFrame to use in SQL queries
# This allows us to reference the data in the MERGE statement using the alias 'data_view'
view_data = data.createOrReplaceTempView("data_view")
table2 = delta_path + "table1"
spark.sql(f"""
  MERGE INTO delta.`{table2}` AS target
  USING data_view AS source
  ON target.id = source.id
  WHEN NOT MATCHED THEN
    INSERT *
""")

In [6]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

delta_table = DeltaTable.forPath(spark, delta_path + "table1")

delta_table.alias("target") \
    .merge(
        data.alias("source"),
        "target.id = source.id"
    ) \
    .whenNotMatchedInsert(values={"id": col("source.id")}) \
    .execute()

In [7]:
df = (spark
        .read
        .format("delta")
        .load(delta_path + "table1")
        .orderBy("id")
      )

df.show()

### Update Delta Lake Table (Delta Lake Transactions)

In [8]:
from delta.tables import *
from pyspark.sql.functions import *


delta_table = DeltaTable.forPath(spark, delta_path + "table1")

spark.sql(f"""
  UPDATE delta.`{delta_path}table1`
  SET id = id + 100
  WHERE id % 2 == 0
""")


In [9]:
# Create multiple modifications to demonstrate vacuum effects
def modify_delta_table(show_history=False):
# Modification 1: Update some records
    spark.sql(f"""
        UPDATE delta.`{delta_path}table1`
        SET id = id + 10
        WHERE id < 105
    """)

    # Modification 2: Delete some records
    spark.sql(f"""
        DELETE FROM delta.`{delta_path}table1`
        WHERE id > 110
    """)

    # Modification 3: Insert new records
    spark.sql(f"""
        INSERT INTO delta.`{delta_path}table1`
        VALUES (200), (201), (202)
    """)

    # Modification 4: Another update
    spark.sql(f"""
        UPDATE delta.`{delta_path}table1`
        SET id = id - 5
        WHERE id >= 200
    """)

    # Show current state
    print("Current table state:")
    delta_table.toDF().orderBy("id").show()

    if show_history:
        # Show table history to see all modifications
        print("\nTable history (showing version accumulation):")
        delta_table.history().select("version", "timestamp", "operation").show(truncate=False)

In [12]:
modify_delta_table(False)

# default retention is 7 days, but we can set it to 0 hours for testing
# RETAIN 0 HOURS option will fail if the following setting is not disabled:
# spark.databricks.delta.retentionDurationCheck.enabled = false

spark.sql(f"""
    VACUUM delta.`{delta_path}table1` 
""")

In [13]:
modify_delta_table(True)
result = spark.sql(f"""
        OPTIMIZE delta.`{delta_path}table1`
""")


In [14]:
result.show(truncate=False)

In [15]:
# Extract and display the optimization metrics in a readable format
metrics = result.select("metrics").collect()[0][0]

print("=" * 80)
print("OPTIMIZE Command Metrics Explanation")
print("=" * 80)

print("\n1. FILES ADDED:")
print(f"   - Number of files added: {metrics['numFilesAdded']}")
print(f"   - Min size: {metrics['filesAdded']['min']} bytes")
print(f"   - Max size: {metrics['filesAdded']['max']} bytes")
print(f"   - Avg size: {metrics['filesAdded']['avg']:.2f} bytes")
print(f"   - Total files: {metrics['filesAdded']['totalFiles']}")
print(f"   - Total size: {metrics['filesAdded']['totalSize']} bytes")

print("\n2. FILES REMOVED:")
print(f"   - Number of files removed: {metrics['numFilesRemoved']}")
print(f"   - Min size: {metrics['filesRemoved']['min']} bytes")
print(f"   - Max size: {metrics['filesRemoved']['max']} bytes")
print(f"   - Avg size: {metrics['filesRemoved']['avg']:.2f} bytes")
print(f"   - Total files: {metrics['filesRemoved']['totalFiles']}")
print(f"   - Total size: {metrics['filesRemoved']['totalSize']} bytes")

print("\n3. PARTITIONS & COLUMNS:")
print(f"   - Partitions optimized: {metrics['partitionsOptimized']}")
print(f"   - Number of table columns: {metrics['numTableColumns']}")
print(f"   - Columns with stats: {metrics['numTableColumnsWithStats']}")

print("\n4. FILES & DATA:")
print(f"   - Total files considered: {metrics['totalConsideredFiles']}")
print(f"   - Total files skipped: {metrics['totalFilesSkipped']}")
print(f"   - Files skipped to reduce write amplification: {metrics['numFilesSkippedToReduceWriteAmplification']}")
print(f"   - Bytes skipped to reduce write amplification: {metrics['numBytesSkippedToReduceWriteAmplification']}")
print(f"   - Number of bins: {metrics['numBins']}")
print(f"   - Number of batches: {metrics['numBatches']}")
print(f"   - Preserve insertion order: {metrics['preserveInsertionOrder']}")

print("\n5. TIMING:")
print(f"   - Start time (ms): {metrics['startTimeMs']}")
print(f"   - End time (ms): {metrics['endTimeMs']}")
print(f"   - Duration (ms): {metrics['endTimeMs'] - metrics['startTimeMs']}")

print("\n6. PARALLELISM:")
print(f"   - Total cluster parallelism: {metrics['totalClusterParallelism']}")
print(f"   - Total scheduled tasks: {metrics['totalScheduledTasks']}")

###  `delete`

In [16]:
# Delete every even value
(delta_table
  .delete(
    condition = expr("id % 2 == 0")
  )
)

(delta_table
  .toDF()
  .orderBy("id")
  .show()
)

### `merge` Delta Lake Table

In [17]:
# Upsert (merge) new data
new_data = spark.range(0, 20)

(delta_table.alias("old_data")
  .merge(
      new_data.alias("new_data"),
      "old_data.id = new_data.id"
      )
  .whenMatchedUpdate(set = { "id": col("new_data.id") })
  .whenNotMatchedInsert(values = { "id": col("new_data.id") })
  .execute()
)

(delta_table
  .toDF()
  .orderBy("id")
  .show(5)
)

### Time Travel: Display the entire history of the above Delta Lake table

In [18]:
# get the full history of the table
delta_table_history = (DeltaTable
                        .forPath(spark, f"{delta_path}table1")
                        .history()
                      )

(delta_table_history
   .select("version", "timestamp", "operation", "operationParameters", "operationMetrics", "engineInfo")
   .show()
)

### Latest version of the Delta Lake table

In [47]:
# get the full history of the table
delta_table_history = (DeltaTable
                        .forPath(spark, f"{delta_path}table1")
                        .history()
                      )

(delta_table_history
   .select("version", "timestamp", "operation", "operationParameters", "operationMetrics", "engineInfo")
   .show()
)

### Latest version of the Delta Lake table

In [49]:
df = (spark
        .read
        .format("delta")
        .load(f"{delta_path}table1")
        .orderBy("id")
      )

df.show(5)

### Time travel to the version `0` of the Delta Lake table using Delta Lake's history feature

In [19]:
df = (spark
        .read
        .format("delta")
        .option("versionAsOf", 0) # we pass an option `versionAsOf` with the required version number we are interested in
        .load(f"{delta_path}table1")
        .orderBy("id")
      )

df.show(5)

### Time travel to the version `3` of the Delta Lake table using Delta Lake's  history feature

In [20]:
df = (spark
        .read
        .format("delta")
        .option("versionAsOf", 3) # we pass an option `versionAsOf` with the required version number we are interested in
        .load(f"{delta_path}table1")
        .orderBy("id")
      )

df.show(5)